Notebook to create normalised tables of the England Census data for input into the AE

In [ ]:
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import os
import re


def populate_data_table(filepath: str) -> pd.DataFrame:
    """
    Populates a data table by merging CSV files in the specified directory.

    Args:
        filepath (str): The path to the directory containing the CSV files.

    Returns:
        pd.DataFrame: The merged data table, or None if no files are found.
    """
    print("Populating data table from files in", filepath)
    data_table = None  # Initialize data_table outside the loop
    # Divide all columns but the first by the second column
    for file in tqdm(os.listdir(filepath), desc="Processing files", unit="file"):
        df = pd.read_csv(filepath + "/" + file, index_col="OA")

        if data_table is None:
            data_table = df  # Initialize data_table with the first file encountered
        else:
            # Merge on index
            data_table = data_table.merge(df, left_index=True, right_index=True, how="outer")
    return data_table

# Prepare census data

In [ ]:
eng_census_raw = populate_data_table("../data/census_data/eng_raw_csvs")
eng_census_raw = eng_census_raw.reindex(sorted(eng_census_raw.columns), axis=1)

eng_census_raw = eng_census_raw.drop(columns=['ts0060001']) #population density, not a census variable
# Remove the Wales only variables
strings_to_remove = ['ts032', 'ts033', 'ts034', 'ts035', 'ts036', 'ts076']
columns_to_keep = [col for col in eng_census_raw.columns if not any(string in col for string in strings_to_remove)]
eng_census_raw = eng_census_raw[columns_to_keep]

# Extract unique prefixes by removing the last four digits
prefixes = set(re.sub(r"\d{4}$", "", col) for col in eng_census_raw.columns if re.match(r".*\d{4}$", col))
# Normalize each tables variables by the total column to get proportions (/ percentages)
for prefix in prefixes:
    base_col = f"{prefix}0001"  # The assumed total column
    group_cols = [col for col in eng_census_raw.columns if col.startswith(prefix)]

    if base_col in eng_census_raw.columns:  # Ensure the base column exists
        eng_census_raw[group_cols] = eng_census_raw[group_cols].div(eng_census_raw[base_col], axis=0)

# Filter columns that end with '001' (the total columns)
columns_to_drop = [col for col in eng_census_raw.columns if col.endswith('001')]
eng_census_raw = eng_census_raw.drop(columns=columns_to_drop)
#round to 3dp 
eng_census_raw = eng_census_raw.round(3)

#drop ts020 and ts055 (they are variables for small subsets (non-uk residents and second homes) so do not have values for most OAs
columns_to_drop = [col for col in eng_census_raw.columns if col.startswith('ts020') or col.startswith('ts055')]
eng_census_raw = eng_census_raw.drop(columns=columns_to_drop)

#print columns with nans
print("Columns with missing values:")
#in eng_census_raw
missing_columns = eng_census_raw.columns[eng_census_raw.isnull().any()].tolist()
print(missing_columns)

#Save the cleaned data (reset index so that OA is a column)
eng_census_raw.reset_index().to_parquet("../data/census_data/engcensus_cleaned_scaled.parquet", index=False)



# Create lookup between 2021 OA and 2011 LSOAs using popweighted centroids
This is used for the IMD error comparisions in notebook 4

In [ ]:


lsoa_2011= gpd.read_file("../data/geofiles/Lower_layer_Super_Output_Areas_Dec_2011_Boundaries_Full_Clipped_BFC_EW_V3_2022_-5365225720680633795.gpkg")
oa_2021 = gpd.read_file("../data/geofiles/Output_Areas_(December_2021)_Boundaries_EW_BFE_(V9)_and_RUC.geojson")

#create lookup between 2021 OA and 2011 LSOAs bounds 
lsoa_2011 = lsoa_2011[['LSOA11CD','geometry']]
oa_2021 = oa_2021[['OA21CD','geometry']]


# Ensure both datasets use the same CRS
oa_2021 = oa_2021.to_crs(lsoa_2011.crs)
# Step 1: intersect OAs with LSOAs
overlap = gpd.overlay(oa_2021, lsoa_2011, how='intersection')

# Step 2: calculate area of each intersected polygon
overlap['overlap_area'] = overlap.geometry.area

# Step 3: for each OA, keep the LSOA with the largest overlap
overlap = overlap.sort_values('overlap_area', ascending=False)
largest_overlap = overlap.drop_duplicates(subset='OA21CD')

# Step 4: assign the LSOA code back to original OA GeoDataFrame
oa_2021 = oa_2021.drop(columns=['LSOA11CD'], errors='ignore')  # clean if already merged
oa_2021 = oa_2021.merge(largest_overlap[['OA21CD', 'LSOA11CD']], on='OA21CD', how='left')

# Check for missing ones (should be zero if geometries are good)
print("Unmatched OAs:", oa_2021['LSOA11CD'].isna().sum())
oa_2021[['OA21CD', 'LSOA11CD']].to_csv("../data/geofiles/lookup_oa2022_lsoa11_EW.csv", index=False)